In [23]:
# Task 11: Execution Plan Analysis
# For at least 3 major queries: Use: df.explain("extended")
# Identify:
# - Exchange
# - BroadcastHashJoin
# - SortMergeJoin
# - WholeStageCodegen
# Explain each component.



In [24]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg
from pyspark.sql.window import Window

In [25]:
spark = SparkSession.builder \
    .appName("ExecutionPlan") \
    .master("yarn") \
    .getOrCreate()

26/02/23 17:25:03 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [26]:
df = spark.read.parquet("hdfs:///data/covid/staging/full_grouped")


### For at least 3 major queries: Use: df.explain("extended")

In [27]:
# 1. Aggregation Query
death_df = df.groupBy("Country/Region") \
             .agg(sum("Deaths").alias("total_deaths"))

death_df.explain("extended")

# Execution Plan Components to Identify:
# - Exchange:
#   Indicates shuffle due to groupBy (wide transformation).
#   Data is hash-partitioned by Country/Region.
#
# - HashAggregate:
#   Performs partial + final aggregation.
#
# - WholeStageCodegen:
#   Spark combines multiple physical operators into a single JVM function
#   for CPU efficiency.


== Parsed Logical Plan ==
'Aggregate ['Country/Region], ['Country/Region, 'sum('Deaths) AS total_deaths#73]
+- Relation [Date#63,Country/Region#64,Confirmed#65L,Deaths#66L,Recovered#67L,Active#68L,New cases#69L,New deaths#70L,New recovered#71L,WHO Region#72] parquet

== Analyzed Logical Plan ==
Country/Region: string, total_deaths: bigint
Aggregate [Country/Region#64], [Country/Region#64, sum(Deaths#66L) AS total_deaths#73L]
+- Relation [Date#63,Country/Region#64,Confirmed#65L,Deaths#66L,Recovered#67L,Active#68L,New cases#69L,New deaths#70L,New recovered#71L,WHO Region#72] parquet

== Optimized Logical Plan ==
Aggregate [Country/Region#64], [Country/Region#64, sum(Deaths#66L) AS total_deaths#73L]
+- Project [Country/Region#64, Deaths#66L]
   +- Relation [Date#63,Country/Region#64,Confirmed#65L,Deaths#66L,Recovered#67L,Active#68L,New cases#69L,New deaths#70L,New recovered#71L,WHO Region#72] parquet

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Country/R

In [28]:
# 2. Join Query
world_df = spark.read.parquet("hdfs:///data/covid/staging/worldometer_data")

joined_df = df.join(
    world_df,
    df["Country/Region"] == world_df["Country/Region"],
    "inner"
)

joined_df.explain("extended")

# Execution Plan Components to Identify:
#
# - Exchange:
#   Shuffle on both sides if Spark chooses SortMergeJoin.
#
# - SortMergeJoin:
#   Used when both datasets are large.
#   Requires shuffle + sort before merge.
#
# - BroadcastHashJoin (if applicable):
#   Appears when smaller dataset is auto-broadcasted.
#   Avoids shuffle on small side.
#
# - WholeStageCodegen:
#   Optimizes execution by generating compiled JVM bytecode.


== Parsed Logical Plan ==
Join Inner, (Country/Region#64 = Country/Region#88)
:- Relation [Date#63,Country/Region#64,Confirmed#65L,Deaths#66L,Recovered#67L,Active#68L,New cases#69L,New deaths#70L,New recovered#71L,WHO Region#72] parquet
+- Relation [Country/Region#88,Continent#89,Population#90L,TotalCases#91L,NewCases#92L,TotalDeaths#93L,NewDeaths#94L,TotalRecovered#95L,NewRecovered#96L,ActiveCases#97L] parquet

== Analyzed Logical Plan ==
Date: date, Country/Region: string, Confirmed: bigint, Deaths: bigint, Recovered: bigint, Active: bigint, New cases: bigint, New deaths: bigint, New recovered: bigint, WHO Region: string, Country/Region: string, Continent: string, Population: bigint, TotalCases: bigint, NewCases: bigint, TotalDeaths: bigint, NewDeaths: bigint, TotalRecovered: bigint, NewRecovered: bigint, ActiveCases: bigint
Join Inner, (Country/Region#64 = Country/Region#88)
:- Relation [Date#63,Country/Region#64,Confirmed#65L,Deaths#66L,Recovered#67L,Active#68L,New cases#69L,New de

In [30]:
# 3. Window Function Query
windowSpec = Window.partitionBy("Country/Region") \
                   .orderBy("Date") \
                   .rowsBetween(-6, 0)

rolling_df = df.withColumn(
    "rolling_avg_recovery",
    avg("Recovered").over(windowSpec)
)

rolling_df.explain("extended")

# Execution Plan Components to Identify:
#
# - Exchange:
#   Occurs if partitioning requires redistribution by Country/Region.
#
# - Sort:
#   Required due to orderBy(Date) inside window.
#
# - Window:
#   Applies rolling aggregation within partition.
#
# - WholeStageCodegen:
#   Combines sort + window operators into optimized execution pipeline.



== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(rolling_avg_recovery, 'avg('Recovered) windowspecdefinition('Country/Region, 'Date ASC NULLS FIRST, specifiedwindowframe(RowFrame, -6, currentrow$())), None)]
+- Relation [Date#63,Country/Region#64,Confirmed#65L,Deaths#66L,Recovered#67L,Active#68L,New cases#69L,New deaths#70L,New recovered#71L,WHO Region#72] parquet

== Analyzed Logical Plan ==
Date: date, Country/Region: string, Confirmed: bigint, Deaths: bigint, Recovered: bigint, Active: bigint, New cases: bigint, New deaths: bigint, New recovered: bigint, WHO Region: string, rolling_avg_recovery: double
Project [Date#63, Country/Region#64, Confirmed#65L, Deaths#66L, Recovered#67L, Active#68L, New cases#69L, New deaths#70L, New recovered#71L, WHO Region#72, rolling_avg_recovery#101]
+- Project [Date#63, Country/Region#64, Confirmed#65L, Deaths#66L, Recovered#67L, Active#68L, New cases#69L, New deaths#70L, New recovered#71L, WHO Region#72, rolling_avg_recovery#101, rolling

In [32]:
spark.stop()